# IADD Traffic Detection - YOLOv11 (Run 5)

Final run on the rebuilt dataset `iadd_subset_v4` (leakage-free by-video split,
70/15/15 by image, rare classes motorcycle/bus/traffic_light balanced and fully kept,
denser val/test so rare-class metrics are no longer noisy).

Changes vs Run 4:
- Dataset iadd_subset_v3 -> iadd_subset_v4.
- Calibrated augmentation to help rare/hard classes and day/night/rain robustness:
  mixup 0.1, hsv_v 0.5 (brightness), degrees 5 (small rotation for camera tilt).
- Trained fresh from COCO (yolo11s), imgsz 1280.
Evaluation: honest mAP at the default threshold + a clean confusion matrix at the
F1-optimal confidence read from the val curve, all in two folders (run5_val, run5_test).

## 1. Check the GPU

In [ ]:
!nvidia-smi

## 2. Install Ultralytics

In [ ]:
!pip install -q ultralytics
import ultralytics; ultralytics.checks()

## 3. Mount Google Drive

The run folder lives on Drive so a limited free session can be continued on another
Google account: share the `iadd_runs` folder and the dataset zip, add a shortcut to
that account's My Drive root, then run every cell - training resumes from `last.pt`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 4. Unzip the dataset into Colab

Colab's local disk is wiped on reconnect, so re-run this cell after any reconnect.

In [ ]:
!rm -rf /content/iadd_subset_v4
!unzip -q "/content/drive/MyDrive/iadd_subset_v4.zip" -d /content
!ls /content/iadd_subset_v4

## 5. Train - fresh, or auto-resume from Drive

If a checkpoint exists on Drive it resumes from the last epoch; otherwise it starts a
fresh Run 5. Augmentation is set explicitly so it is documented and reproducible.

In [ ]:
from ultralytics import YOLO
import os

MODEL    = 'yolo11s.pt'
IMGSZ    = 1280
BATCH    = 12
EPOCHS   = 150
PATIENCE = 20

DRIVE_PROJECT = '/content/drive/MyDrive/iadd_runs'
RUN_NAME      = 'run5'
DATA          = '/content/iadd_subset_v4/data.yaml'
CKPT          = f'{DRIVE_PROJECT}/{RUN_NAME}/weights/last.pt'

if os.path.exists(CKPT):
    print('checkpoint found -> resuming:', CKPT)
    model = YOLO(CKPT)
    results = model.train(resume=True)
else:
    print('no checkpoint -> fresh run:', MODEL, '| imgsz', IMGSZ, '| batch', BATCH)
    model = YOLO(MODEL)
    results = model.train(
        data=DATA, epochs=EPOCHS, patience=PATIENCE, imgsz=IMGSZ, batch=BATCH,
        project=DRIVE_PROJECT, name=RUN_NAME, exist_ok=True, plots=True,
        # calibrated augmentation (rest left at Ultralytics defaults)
        mixup=0.1,       # blend image pairs -> rare classes seen in new contexts
        hsv_v=0.5,       # brightness jitter -> day / night / rain robustness
        degrees=5.0,     # small rotation -> camera tilt from road vibration
    )

## 6. Progress so far (optional)

In [ ]:
import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/iadd_runs/run5/results.csv')
df.columns = [c.strip() for c in df.columns]
print('epochs done:', len(df))
df[['epoch', 'metrics/mAP50(B)', 'metrics/mAP50-95(B)',
    'metrics/precision(B)', 'metrics/recall(B)']].tail(10)

## 7. Validate - honest metrics + operating confidence + clean matrix

Runs validation at the default threshold (COCO-standard mAP, per-class table, and the
P/R/F1/PR curves), reads the F1-optimal confidence from the val F1 curve, then writes a
clean confusion matrix at that confidence into the same `run5_val` folder.

In [ ]:
from ultralytics import YOLO
import numpy as np, os, shutil

DATA = '/content/iadd_subset_v4/data.yaml'
RUNS = '/content/drive/MyDrive/iadd_runs'
best = YOLO(f'{RUNS}/run5/weights/best.pt')

# honest metrics + curves (threshold-independent mAP)
mv = best.val(data=DATA, split='val', plots=True, project=RUNS, name='run5_val', exist_ok=True)
print('VAL  mAP50=%.4f  mAP50-95=%.4f' % (mv.box.map50, mv.box.map))

# operating confidence = highest mean F1, read from the val F1 curve (never test)
CONF = 0.25
for x, y, _, ylabel in mv.curves_results:
    if ylabel == 'F1':
        px = np.asarray(x); y = np.asarray(y)
        f1 = y.mean(0) if y.ndim > 1 else y
        CONF = round(float(px[f1.argmax()]), 3)
print('operating confidence (from val F1 curve) =', CONF)

# clean confusion matrix at CONF -> overwrite the messy default one in run5_val
t = best.val(data=DATA, split='val', conf=CONF, plots=True, project=RUNS, name='run5_val_tmp', exist_ok=True)
for f in ('confusion_matrix.png', 'confusion_matrix_normalized.png'):
    shutil.copy(f'{RUNS}/run5_val_tmp/{f}', f'{RUNS}/run5_val/{f}')
shutil.rmtree(f'{RUNS}/run5_val_tmp')
print('run5_val ready: honest curves + confusion matrix at conf', CONF)

## 8. Test - honest metrics + clean matrix at the same confidence

The test videos were never seen in training (leakage-free split), so this is the
number to report. The confusion matrix uses the same operating confidence as val.

In [ ]:
mt = best.val(data=DATA, split='test', plots=True, project=RUNS, name='run5_test', exist_ok=True)
print('TEST mAP50=%.4f  mAP50-95=%.4f' % (mt.box.map50, mt.box.map))

t = best.val(data=DATA, split='test', conf=CONF, plots=True, project=RUNS, name='run5_test_tmp', exist_ok=True)
for f in ('confusion_matrix.png', 'confusion_matrix_normalized.png'):
    shutil.copy(f'{RUNS}/run5_test_tmp/{f}', f'{RUNS}/run5_test/{f}')
shutil.rmtree(f'{RUNS}/run5_test_tmp')
print('run5_test ready: honest curves + confusion matrix at conf', CONF)

## 9. Demo predictions - saved to run5/demo

In [ ]:
import glob

imgs = sorted(glob.glob('/content/iadd_subset_v4/images/test/*.jpg'))[:16]
best.predict(imgs, conf=CONF, save=True, project=f'{RUNS}/run5', name='demo', exist_ok=True)
print('demo images saved -> iadd_runs/run5/demo  (conf=%.3f)' % CONF)

---
Results on Drive under `iadd_runs/`:
- `run5/` - training, weights (`best.pt`, `last.pt`), and `demo/`
- `run5_val/`, `run5_test/` - honest curves + clean confusion matrix at the F1-optimal conf